# 11 — XGBoost ile doğrusal olmayan post-hoc confidence tahmini

**Veri:** `validated_v007` / `feature_assembly_v2_repair1` / `CLEAN_DEV` (970 soru/model).  
**Karşılaştırma:** Stage 10'da dondurulmuş Logistic CORE OOF tahminleri ↔ XGBoost CORE.  
**Soru:** Aynı frozen confidence sinyalleriyle doğrusal olmayan bir model, held-out sorularda daha düşük olasılık hatası üretiyor mu?

Bu notebook **yeni LLM inference yapmaz**; Stage 08'in frozen özellikleri ve Stage 10'un frozen fold/OOF sonuçlarını kullanır. Aynı sorunun üç tekrarını **bağımsız örnek saymaz**. Sonuçlar development OOF'dur; bağımsız final test değildir.

**Önceden tanımlanmış deney kolları:**

- `logistic_core_stage10`: frozen Stage 10 referans, yeniden fit edilmez.
- `xgb_core_nested`: ana yeni yöntem; CORE 8 özellik, dış 5-fold × 3 tekrar; hiperparametre seçimi yalnız eğitim bölmesindeki 3-fold inner CV ile.
- `xgb_core_fixed`: tek sabit konfigürasyon, tuning'e duyarlılık.
- `xgb_core_minus_consistency`: CORE'dan yalnız `self_consistency` çıkarılır.
- `xgb_extended`: 27 özellik; **CORE için seçilen aynı dış-fold hiperparametreleri** kullanılır. Bu, EXTENDED için ayrı optimizasyon değildir.

XGBoost'un `predict_proba` çıktısı kendiliğinden kalibre edilmiş olasılık garanti etmez. Brier/NLL/ECE ile ayrıca değerlendirilecektir. Önerilen ana metrik **Brier**, diğerleri tamamlayıcıdır.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install 'xgboost>=2.0,<3.2' 'scikit-learn>=1.5,<1.9' 'pyarrow>=15,<25'

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 7.8 MB/s eta 0:00:00


## 1. Import, dosya kimliği ve frozen provenance

İlk hücreler bütünlük uyuşmazlığında işlemi durdurur. Stage 10 dosyalarının üzerine yazılmaz.

In [2]:
import hashlib, json, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

ROOT = Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
VERSION = 'validated_v007'
ASSEMBLY = 'feature_assembly_v2_repair1'
BASE = ROOT / 'features' / 'assembled' / VERSION
M08 = BASE / 'stage08_manifest_v2_repair1.json'
SCHEMA = BASE / 'feature_schema_v1.json'
MATRIX = BASE / 'extended_model_matrix_v2_repair1.parquet'
REPAIR = ROOT / 'results' / 'stage05' / f'{VERSION}_stage05_self_confidence_repair_v1.json'

def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    with open(path, encoding='utf-8') as handle:
        return json.load(handle)

def require(path):
    if not path.is_file():
        raise FileNotFoundError(path)

def immutable_bytes(path, blob):
    if path.exists():
        if path.read_bytes() != blob:
            raise RuntimeError(f'Immutable output conflict: {path}')
        return
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_bytes(blob)
    os.replace(tmp, path)

def immutable_csv(df, path):
    immutable_bytes(path, df.to_csv(index=False).encode('utf-8'))

for p in [M08, SCHEMA, MATRIX, REPAIR]:
    require(p)
m08, schema, repair = read_json(M08), read_json(SCHEMA), read_json(REPAIR)
assert m08['dataset_version'] == schema['dataset_version'] == VERSION
assert m08['assembly_version'] == schema['assembly_version'] == ASSEMBLY
assert sha_file(MATRIX) == m08['outputs']['extended_model_matrix_sha256']
assert sha_file(SCHEMA) == m08['outputs']['feature_schema_file_sha256']
assert schema['schema_sha256'] == m08['feature_schema_sha256']
assert sha_file(REPAIR) == m08['upstream']['stage05_self_confidence_repair_manifest_sha256']
assert repair['base_stage05_completion_sha256'] == m08['upstream']['stage05_completion_sha256']
repair_sidecar = Path(repair['repair_file'])
require(repair_sidecar)
assert sha_file(repair_sidecar) == repair['repair_file_sha256']
assert sha_file(repair_sidecar) == m08['upstream']['stage05_self_confidence_repair_file_sha256']

STAGE10_DIR = (
    ROOT / 'results' / 'stage10' / VERSION
    / ('stage08_' + sha_file(M08)[:12] + '_repair1')
)
M10 = STAGE10_DIR / 'stage10_dev_manifest_v1.json'
require(M10)
m10 = read_json(M10)

for name, item in m10['outputs'].items():
    p = Path(item['path'])
    require(p)
    if sha_file(p) != item['sha256']:
        raise RuntimeError(f'Stage10 frozen output SHA mismatch: {name}')
assert m10['created_from_stage08_sha256'] == sha_file(M08)
contract10 = read_json(Path(m10['outputs']['contract']['path']))
assert contract10['stage08_manifest_sha256'] == sha_file(M08)
assert contract10['stage05_repair_manifest_sha256'] == sha_file(REPAIR)
assert contract10['dataset_version'] == VERSION
assert contract10['assembly_version'] == ASSEMBLY
assert contract10['scope'] == 'CLEAN_DEV: exclude_from_final_test==0'

SEEDS = list(contract10['seeds'])
N_SPLITS = int(contract10['n_splits'])
EPS = float(contract10['nll_clip_eps'])
ECE_BINS = int(contract10['ece_equal_width_bins'])
assert SEEDS == [1101, 2202, 3303] and N_SPLITS == 5
assert ECE_BINS == 10 and EPS == 1e-6
assert contract10['logistic_feature_contract']['core'] == schema['core_features']

full = pd.read_parquet(MATRIX)
core = list(schema['core_features'])
extended = list(schema['extended_features'])
target = str(schema['target'])
assert len(full) == 3000 and full['model_id'].nunique() == 3
assert not full.duplicated(['question_id', 'model_id']).any()
assert len(core) == 8 and len(extended) == 27
assert target not in extended and 'manual_error_type' not in extended
assert core == contract10['logistic_feature_contract']['core']
assert set(full[target].unique()) == {0, 1}
for f in extended:
    if f not in full.columns or not np.isfinite(pd.to_numeric(full[f], errors='coerce').to_numpy(dtype=float)).all():
        raise RuntimeError(f'Missing/nonfinite predictor: {f}')
clean = full.loc[full['exclude_from_final_test'].astype(int) == 0].copy()
assert len(clean) == 2910
assert (clean.groupby('model_id').size() == 970).all()
assert (clean.groupby('model_id')['question_id'].nunique() == 970).all()
assert not clean['used_in_model_selection'].astype(bool).any()

stage10_oof_path = Path(m10['outputs']['oof_predictions']['path'])
stage10_metrics_path = Path(m10['outputs']['per_repeat_metrics']['path'])
stage10_oof = pd.read_parquet(stage10_oof_path)
stage10_metrics = pd.read_csv(stage10_metrics_path)
assert not stage10_oof.duplicated(['model_id', 'question_id', 'repeat', 'method']).any()
assert m10['n_oof_prediction_rows'] == len(stage10_oof)
assert (stage10_oof.groupby(['model_id','repeat','method']).size() == 970).all()

print('Stage08 repair-aware provenance: PASS')
print('Stage10 frozen SHA/provenance: PASS')
print('CLEAN_DEV: 970 × 3 =', len(clean))
print('Stage10 OOF rows:', len(stage10_oof))
print('CORE:', core)
print('Versions:', 'xgboost', xgboost.__version__, '| sklearn', sklearn.__version__)

Stage08 repair-aware provenance: PASS
Stage10 frozen SHA/provenance: PASS
CLEAN_DEV: 970 × 3 = 2910
Stage10 OOF rows: 139680
CORE: ['mean_logprob', 'min_logprob', 'mean_token_entropy_normalized', 'output_content_token_count', 'self_confidence_scaled', 'verification_p_correct', 'self_consistency', 'unknown_response']
Versions: xgboost 3.1.3 | sklearn 1.6.1


## 2. Önceden belirlenmiş XGBoost deney sözleşmesi

İç CV aday seti **dış fold sonuçlarına bakılmadan** tanımlanır. Dört küçük model (derinlik 1/2 × 60/120 ağaç) eğitim fold'undaki 3-fold Brier ile karşılaştırılır. Eşitlikte daha küçük model seçilir. İç CV dışındaki veriler hiperparametre seçimine girmez.

`xgb_core_fixed` adaylardan en büyük olanı (`max_depth=2, n_estimators=120`) kullanır. `xgb_extended` ve `xgb_core_minus_consistency` her dış fold için `xgb_core_nested` tarafından eğitim verisinde seçilen hiperparametreleri yeniden kullanır; bunlar **özellik seti ablation'larıdır, kendileri için optimize edilmiş XGBoost modelleri değildir**.

Sözleşme dosyası **fit işleminden önce** salt yeni bir sürüm yoluna yazılır. Parametreleri sonuca göre değiştirmek istersen önce sürüm numarasını artır ve yeni bir keşif deneyi olarak belgele.

In [3]:
EXPERIMENT_VERSION = 'xgb_nested_v1'
XGB_METHODS = [
    'xgb_core_nested',
    'xgb_core_fixed',
    'xgb_core_minus_consistency',
    'xgb_extended',
]
REFERENCE_METHODS = ['core', 'train_prevalence', 'raw_verification']
CORE_MINUS_CONSISTENCY = [f for f in core if f != 'self_consistency']
assert len(CORE_MINUS_CONSISTENCY) == 7

# Tamamen önceden tanımlanmış adaylar ve tie-break.
CANDIDATES = [
    {'max_depth': depth, 'n_estimators': trees}
    for depth in [1, 2]
    for trees in [60, 120]
]
FIXED_CANDIDATE = {'max_depth': 2, 'n_estimators': 120}
INNER_SPLITS = 3
XGB_RANDOM_STATE = 17011

XGB_SHARED = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'learning_rate': 0.05,
    'min_child_weight': 5,
    'subsample': 0.85,
    'colsample_bytree': 0.9,
    'reg_lambda': 5.0,
    'reg_alpha': 0.0,
    'tree_method': 'hist',
    'n_jobs': 2,
    'random_state': XGB_RANDOM_STATE,
}

BOOTSTRAP_REPS = 4000
BOOTSTRAP_SEED = 20260920

OUT = (
    ROOT / 'results' / 'stage11' / VERSION
    / ('stage10_' + sha_file(M10)[:12] + '_' + EXPERIMENT_VERSION)
)
OUT.mkdir(parents=True, exist_ok=True)

PATHS = {
    'contract': OUT / 'experiment_contract_v1.json',
    'inner_cv': OUT / 'inner_cv_candidate_scores_v1.csv',
    'selection': OUT / 'outer_fold_selected_hyperparameters_v1.csv',
    'oof': OUT / 'xgb_oof_predictions_v1.parquet',
    'metrics': OUT / 'per_repeat_metrics_v1.csv',
    'summary': OUT / 'repeated_oof_summary_v1.csv',
    'paired': OUT / 'paired_question_bootstrap_v1.csv',
    'risk_coverage': OUT / 'risk_coverage_v1.csv',
    'manifest': OUT / 'stage11_dev_manifest_v1.json',
}

contract11 = {
    'stage': '11_nonlinear_confidence_dev',
    'dataset_version': VERSION,
    'assembly_version': ASSEMBLY,
    'stage08_manifest_sha256': sha_file(M08),
    'stage05_repair_manifest_sha256': sha_file(REPAIR),
    'stage10_manifest_sha256': sha_file(M10),
    'stage10_frozen_oof_sha256': sha_file(stage10_oof_path),
    'stage10_frozen_metrics_sha256': sha_file(stage10_metrics_path),
    'scope': 'CLEAN_DEV: 970 unique questions per model',
    'n_models': 3,
    'outer_n_splits': N_SPLITS,
    'outer_seeds': SEEDS,
    'outer_folds_exactly_match_stage10': True,
    'inner_n_splits': INNER_SPLITS,
    'inner_split': 'StratifiedKFold(shuffle=True, seed=17011+100*repeat+fold)',
    'inner_selection_metric': 'mean_validation_brier',
    'inner_tie_break': 'mean_brier,n_estimators,max_depth',
    'xgb_shared_parameters': XGB_SHARED,
    'xgb_hyperparameter_candidates': CANDIDATES,
    'xgb_fixed_candidate': FIXED_CANDIDATE,
    'xgb_methods': XGB_METHODS,
    'reference_methods_from_stage10': REFERENCE_METHODS,
    'core_features': core,
    'core_minus_consistency_features': CORE_MINUS_CONSISTENCY,
    'extended_features': extended,
    'secondary_methods_reuse_core_selected_hyperparameters': True,
    'nll_clip_eps': EPS,
    'ece_equal_width_bins': ECE_BINS,
    'bootstrap_reps': BOOTSTRAP_REPS,
    'bootstrap_seed': BOOTSTRAP_SEED,
    'primary_metric': 'Brier',
    'status': 'development_oof_not_independent_final_test',
    'versions': {'xgboost': xgboost.__version__, 'sklearn': sklearn.__version__,
                 'numpy': np.__version__, 'pandas': pd.__version__},
}
immutable_bytes(PATHS['contract'], json.dumps(
    contract11, ensure_ascii=False, sort_keys=True, indent=2
).encode('utf-8'))
print('PRE-REGISTERED DEV CONTRACT:', PATHS['contract'])
print('Candidate grid:', CANDIDATES)
print('Output directory:', OUT)

PRE-REGISTERED DEV CONTRACT: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage11/validated_v007/stage10_ada008b24140_xgb_nested_v1/experiment_contract_v1.json
Candidate grid: [{'max_depth': 1, 'n_estimators': 60}, {'max_depth': 1, 'n_estimators': 120}, {'max_depth': 2, 'n_estimators': 60}, {'max_depth': 2, 'n_estimators': 120}]
Output directory: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage11/validated_v007/stage10_ada008b24140_xgb_nested_v1


## 3. Stage 10 dış fold'larını tam olarak yeniden üret ve key/label eşleşmesini doğrula

`same-fold` kontrolü yalnız satır sayılarını değil, **her model–soru–tekrar için fold ID ve hedef etiketini** karşılaştırır. Stage 10 referans tahminleri dondurulmuş dosyadan okunur; Logistic CORE yeniden eğitilmez.

In [4]:
outer_splits = {}
outer_audit_rows = []

for model_id, part in clean.groupby('model_id', sort=True):
    part = part.sort_values('question_id').reset_index(drop=True)
    y = part[target].astype(int).to_numpy()
    assert len(part) == 970 and len(np.unique(y)) == 2
    frozen = stage10_oof.loc[
        (stage10_oof['model_id'] == model_id)
        & (stage10_oof['method'] == 'core'),
        ['question_id', 'repeat', 'fold', 'manual_is_correct'],
    ].copy()
    assert len(frozen) == 970 * len(SEEDS)
    frozen = frozen.set_index(['question_id', 'repeat'])
    assert not frozen.index.has_duplicates

    for rep, seed in enumerate(SEEDS):
        cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=1, random_state=seed)
        visited = np.zeros(len(part), dtype=int)
        for fold, (tr, te) in enumerate(cv.split(np.zeros(len(part)), y)):
            assert not np.intersect1d(tr, te).size
            assert len(np.unique(y[tr])) == len(np.unique(y[te])) == 2
            assert len(te) == 194
            visited[te] += 1
            keys = [(qid, rep) for qid in part.iloc[te]['question_id']]
            stored = frozen.loc[keys]
            if not (stored['fold'].astype(int).to_numpy() == fold).all():
                raise RuntimeError(f'Stage10 fold mismatch: {model_id} {rep} {fold}')
            if not np.array_equal(stored['manual_is_correct'].astype(int).to_numpy(), y[te]):
                raise RuntimeError(f'Stage10 label mismatch: {model_id} {rep} {fold}')
            outer_splits[(model_id, rep, fold)] = (tr, te)
            outer_audit_rows.append({
                'model_id': model_id, 'repeat': rep, 'fold': fold,
                'n_train': len(tr), 'n_test': len(te),
                'n_train_correct': int(y[tr].sum()), 'n_test_correct': int(y[te].sum()),
                'n_question_overlap': int(np.intersect1d(
                    part.iloc[tr]['question_id'], part.iloc[te]['question_id']
                ).size),
            })
        assert (visited == 1).all()

outer_audit = pd.DataFrame(outer_audit_rows)
assert len(outer_audit) == 3 * len(SEEDS) * N_SPLITS
assert outer_audit['n_question_overlap'].max() == 0
print('STAGE 10 / STAGE 11 SAME-FOLD + LABEL CONTRACT: PASS')
display(outer_audit.groupby('model_id', as_index=False).agg(
    outer_folds=('fold', 'count'), min_test=('n_test', 'min'),
    max_test=('n_test', 'max'), max_overlap=('n_question_overlap', 'max')
))

STAGE 10 / STAGE 11 SAME-FOLD + LABEL CONTRACT: PASS


,model_id,outer_folds,min_test,max_test,max_overlap
0,Qwen/Qwen3-4B,15,194,194,0
1,google/gemma-3-4b-it,15,194,194,0
2,microsoft/Phi-4-mini-instruct,15,194,194,0


## 4. Inner CV yardımcıları ve sentetik birim testi

Her outer train'in içinde yalnızca 776 eğitim sorusu bulunur. İç fold'lar bu alt küme üzerinde oluşturulur. Inner train/validation ayrımı ve XGBoost random seed sabittir. Test, önce gerçek veriye fit etmeden yardımcı fonksiyonların boyut ve olasılık kontrolünü yapar.

In [5]:
def new_xgb(candidate):
    return XGBClassifier(**XGB_SHARED, **candidate)

def predict_xgb(train, test, features, candidate):
    x_train = train[features].to_numpy(dtype=np.float32)
    x_test = test[features].to_numpy(dtype=np.float32)
    y_train = train[target].astype(int).to_numpy()
    assert np.isfinite(x_train).all() and np.isfinite(x_test).all()
    assert len(np.unique(y_train)) == 2
    model = new_xgb(candidate)
    model.fit(x_train, y_train, verbose=False)
    p = np.asarray(model.predict_proba(x_test)[:, 1], dtype=float)
    if p.shape != (len(test),) or not np.isfinite(p).all():
        raise RuntimeError('Invalid XGBoost probability output')
    if (p < 0).any() or (p > 1).any():
        raise RuntimeError('XGBoost probability outside [0,1]')
    return p

def inner_tune_core(outer_train, rep, fold):
    y_outer = outer_train[target].astype(int).to_numpy()
    inner_seed = 17011 + 100 * rep + fold
    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS, shuffle=True, random_state=inner_seed
    )
    # Tüm adaylar kesinlikle aynı inner folds üzerinde ölçülür.
    inner_folds = list(inner_cv.split(np.zeros(len(outer_train)), y_outer))
    rows = []
    for candidate in CANDIDATES:
        fold_losses = []
        for inner_fold, (itr, iva) in enumerate(inner_folds):
            assert not np.intersect1d(itr, iva).size
            inner_training = outer_train.iloc[itr]
            inner_validation = outer_train.iloc[iva]
            assert len(np.unique(y_outer[itr])) == 2
            assert len(np.unique(y_outer[iva])) == 2
            p = predict_xgb(inner_training, inner_validation, core, candidate)
            loss = float(brier_score_loss(y_outer[iva], p))
            fold_losses.append(loss)
            rows.append({
                'repeat': rep, 'outer_fold': fold,
                'max_depth': candidate['max_depth'],
                'n_estimators': candidate['n_estimators'],
                'inner_fold': inner_fold,
                'inner_n_train': len(itr), 'inner_n_validation': len(iva),
                'inner_brier': loss,
            })
        assert len(fold_losses) == INNER_SPLITS
    tmp = pd.DataFrame(rows)
    avg = tmp.groupby(['max_depth', 'n_estimators'], as_index=False)['inner_brier'].mean()
    # Kesin deterministik tie-break: Brier, ağaç sayısı, derinlik.
    best = avg.sort_values(['inner_brier', 'n_estimators', 'max_depth'],
                           kind='mergesort').iloc[0]
    selected = {'max_depth': int(best['max_depth']),
                'n_estimators': int(best['n_estimators'])}
    return selected, float(best['inner_brier']), tmp

# Smoke test: gerçek data label'ına erişmeden küçük synthetic fit.
rng_test = np.random.default_rng(11)
smoke_x = rng_test.normal(size=(45, len(core)))
smoke_y = np.array([0, 1] * 22 + [0], dtype=int)
smoke = pd.DataFrame(smoke_x, columns=core)
smoke[target] = smoke_y
smoke_p = predict_xgb(smoke.iloc[:36], smoke.iloc[36:], core, CANDIDATES[0])
assert smoke_p.shape == (9,) and np.isfinite(smoke_p).all()
assert ((smoke_p >= 0) & (smoke_p <= 1)).all()
print('XGBOOST HELPER SMOKE TEST: PASS')

XGBOOST HELPER SMOKE TEST: PASS


## 5. Nested XGBoost: 3 model × 3 tekrar × 5 dış fold

Bu hücre inner-CV XGBoost fit'lerini çalıştırır. **LLM çağrısı yapmaz.** CPU Colab'da birkaç dakika sürebilir; ilerlemeyi her dış fold sonunda yazdırır. Durdurup tekrar çalıştırırsan aynı deterministik hesaplamayı baştan yapar; ana Stage 10 artefaktları değişmez.

In [6]:
xgb_rows = []
inner_score_frames = []
selection_rows = []

for model_id, part in clean.groupby('model_id', sort=True):
    part = part.sort_values('question_id').reset_index(drop=True)
    y = part[target].astype(int).to_numpy()
    print('Starting:', model_id, flush=True)

    for rep in range(len(SEEDS)):
        for fold in range(N_SPLITS):
            tr, te = outer_splits[(model_id, rep, fold)]
            training, heldout = part.iloc[tr], part.iloc[te]
            # Hiperparametre seçimi test fold'una hiçbir zaman erişmez.
            selected, inner_brier, scores = inner_tune_core(training, rep, fold)
            scores.insert(0, 'model_id', model_id)
            inner_score_frames.append(scores)
            selection_rows.append({
                'model_id': model_id, 'repeat': rep, 'fold': fold,
                'selected_max_depth': selected['max_depth'],
                'selected_n_estimators': selected['n_estimators'],
                'selected_inner_mean_brier': inner_brier,
                'n_outer_train': len(tr), 'n_outer_test': len(te),
            })

            features_by_method = {
                'xgb_core_nested': (core, selected),
                'xgb_core_fixed': (core, FIXED_CANDIDATE),
                'xgb_core_minus_consistency': (CORE_MINUS_CONSISTENCY, selected),
                'xgb_extended': (extended, selected),
            }
            for method, (features, hyperparameters) in features_by_method.items():
                p = predict_xgb(training, heldout, features, hyperparameters)
                for row_idx, prob in zip(te, p):
                    xgb_rows.append({
                        'model_id': model_id,
                        'question_id': part.loc[row_idx, 'question_id'],
                        'repeat': rep, 'fold': fold, 'method': method,
                        'manual_is_correct': int(y[row_idx]),
                        'p_correct': float(prob),
                    })
            print(f'  repeat={rep} fold={fold} selected={selected} '
                  f'inner_Brier={inner_brier:.5f}', flush=True)
    print('Completed:', model_id, flush=True)

xgb_oof = pd.DataFrame(xgb_rows)
inner_scores = pd.concat(inner_score_frames, ignore_index=True)
selection_df = pd.DataFrame(selection_rows)

assert len(selection_df) == 3 * len(SEEDS) * N_SPLITS
assert len(inner_scores) == len(selection_df) * len(CANDIDATES) * INNER_SPLITS
assert len(xgb_oof) == len(clean) * len(SEEDS) * len(XGB_METHODS)
assert not xgb_oof.duplicated(['model_id', 'question_id', 'repeat', 'method']).any()
assert (xgb_oof.groupby(['model_id','repeat','method']).size() == 970).all()
assert (xgb_oof.groupby(['model_id','question_id','repeat'])['fold'].nunique() == 1).all()
assert (xgb_oof.groupby(['model_id','question_id','repeat'])['manual_is_correct'].nunique() == 1).all()

# Stage10 ile key, label ve fold eşitliğini yeniden doğrula.
frozen_core = stage10_oof.loc[
    stage10_oof['method'] == 'core',
    ['model_id','question_id','repeat','fold','manual_is_correct']
]
joined = xgb_oof.merge(
    frozen_core,
    on=['model_id','question_id','repeat'],
    how='left', validate='many_to_one', suffixes=('_new','_stage10'),
)
assert len(joined) == len(xgb_oof)
assert (joined['fold_new'] == joined['fold_stage10']).all()
assert (joined['manual_is_correct_new'] == joined['manual_is_correct_stage10']).all()

print('XGBOOST OOF COVERAGE + STAGE10 SAME-FOLD: PASS', xgb_oof.shape)
display(selection_df.groupby(['model_id','selected_max_depth','selected_n_estimators'],
                             as_index=False).size().rename(columns={'size':'n_outer_folds'}))

Starting: Qwen/Qwen3-4B
  repeat=0 fold=0 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.11620
  repeat=0 fold=1 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.11367
  repeat=0 fold=2 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.11223
  repeat=0 fold=3 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.10088
  repeat=0 fold=4 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.10724
  repeat=1 fold=0 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.10657
  repeat=1 fold=1 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.11231
  repeat=1 fold=2 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.11906
  repeat=1 fold=3 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.10942
  repeat=1 fold=4 selected={'max_depth': 2, 'n_estimators': 120} inner_Brier=0.10931
  repeat=2 fold=0 selected={'max_depth': 1, 'n_estimators': 120} inner_Brier=0.11041
  repeat=2 fold=1 selected={'max_depth': 

,model_id,selected_max_depth,selected_n_estimators,n_outer_folds
0,Qwen/Qwen3-4B,1,120,1
1,Qwen/Qwen3-4B,2,120,14
2,google/gemma-3-4b-it,1,120,2
3,google/gemma-3-4b-it,2,120,13
4,microsoft/Phi-4-mini-instruct,1,120,1
5,microsoft/Phi-4-mini-instruct,2,120,14


## 6. Aynı OOF sorularında metrikler ve Stage 10 referansıyla bire bir kontrol

Stage 10'daki `core`, `train_prevalence`, `raw_verification` tahminleri dosyadan okunur. `logistic_core_stage10` yalnız karşılaştırma adıdır.  Her tekrar için **970 soru** üzerinden Brier, NLL, AUROC, AUPRC ve ECE hesaplanır; tekrarlar tek bir 2910 örnekli test olarak havuzlanmaz.

In [7]:
reference_oof = stage10_oof.loc[
    stage10_oof['method'].isin(REFERENCE_METHODS)
].copy()
reference_oof['method'] = reference_oof['method'].map({
    'core': 'logistic_core_stage10',
    'train_prevalence': 'train_prevalence_stage10',
    'raw_verification': 'raw_verification_stage10',
})
all_oof = pd.concat([reference_oof, xgb_oof], ignore_index=True)
assert not all_oof.duplicated(['model_id','question_id','repeat','method']).any()
assert (all_oof.groupby(['model_id','repeat','method']).size() == 970).all()
assert (all_oof.groupby(['model_id','question_id','repeat'])['fold'].nunique() == 1).all()
assert (all_oof.groupby(['model_id','question_id','repeat'])['manual_is_correct'].nunique() == 1).all()

def calc_ece(y, p, bins=ECE_BINS):
    idx = np.minimum((np.clip(p, 0, 1) * bins).astype(int), bins - 1)
    return float(sum(
        (int((idx == b).sum()) / len(y)) *
        abs(float(p[idx == b].mean()) - float(y[idx == b].mean()))
        for b in range(bins) if (idx == b).any()
    ))

def calc_metrics(y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    clipped = np.clip(p, EPS, 1 - EPS)
    return {
        'n': len(y), 'n_correct': int(y.sum()),
        'prevalence': float(y.mean()),
        'auroc': float(roc_auc_score(y, p)),
        'auprc': float(average_precision_score(y, p)),
        'brier': float(brier_score_loss(y, p)),
        'nll': float(log_loss(y, clipped, labels=[0, 1])),
        'ece_10_bins': calc_ece(y, p),
        'mean_confidence': float(p.mean()),
    }

metric_rows = []
for (model_id, rep, method), block in all_oof.groupby(
    ['model_id','repeat','method'], sort=True
):
    block = block.sort_values('question_id')
    result = calc_metrics(block['manual_is_correct'], block['p_correct'])
    metric_rows.append({'model_id':model_id,'repeat':int(rep),'method':method, **result})
metrics11 = pd.DataFrame(metric_rows)

# Stage 10 metriklerini (aynı OOF veri ve aynı tanımlarla) tekrar elde etmeliyiz.
for stage10_method, stage11_method in [
    ('core','logistic_core_stage10'),
    ('train_prevalence','train_prevalence_stage10'),
    ('raw_verification','raw_verification_stage10'),
]:
    old = stage10_metrics.loc[stage10_metrics['method'] == stage10_method]
    new = metrics11.loc[metrics11['method'] == stage11_method]
    compared = new.merge(old, on=['model_id','repeat'], validate='one_to_one',
                         suffixes=('_stage11','_stage10'))
    assert len(compared) == 3 * len(SEEDS)
    for metric in ['auroc','auprc','brier','nll','ece_10_bins']:
        difference = np.abs(compared[f'{metric}_stage11'] - compared[f'{metric}_stage10'])
        if difference.max() > 1e-10:
            raise RuntimeError(f'Stage10 baseline metric mismatch: {stage10_method}/{metric}')

summary11 = metrics11.groupby(['model_id','method'], as_index=False).agg(
    n_repeats=('repeat','nunique'),
    mean_auroc=('auroc','mean'), sd_auroc=('auroc','std'),
    mean_auprc=('auprc','mean'),
    mean_brier=('brier','mean'), sd_brier=('brier','std'),
    mean_nll=('nll','mean'), sd_nll=('nll','std'),
    mean_ece_10_bins=('ece_10_bins','mean'),
)
base = metrics11.loc[metrics11['method'] == 'train_prevalence_stage10',
                     ['model_id','repeat','brier','nll']].rename(
    columns={'brier':'baseline_brier','nll':'baseline_nll'}
)
with_baseline = metrics11.merge(base, on=['model_id','repeat'], validate='many_to_one')
with_baseline['delta_brier_vs_prevalence'] = with_baseline['brier'] - with_baseline['baseline_brier']
with_baseline['delta_nll_vs_prevalence'] = with_baseline['nll'] - with_baseline['baseline_nll']
deltas = with_baseline.groupby(['model_id','method'], as_index=False).agg(
    mean_delta_brier_vs_prevalence=('delta_brier_vs_prevalence','mean'),
    mean_delta_nll_vs_prevalence=('delta_nll_vs_prevalence','mean'),
)
summary11 = summary11.merge(deltas, on=['model_id','method'],validate='one_to_one')

print('STAGE10 BASELINE REPRODUCTION: PASS')
display(summary11[[
    'model_id','method','n_repeats','mean_auroc','mean_auprc',
    'mean_brier','mean_nll','mean_ece_10_bins',
    'mean_delta_brier_vs_prevalence',
]])

STAGE10 BASELINE REPRODUCTION: PASS


,model_id,method,n_repeats,mean_auroc,mean_auprc,mean_brier,mean_nll,mean_ece_10_bins,mean_delta_brier_vs_prevalence
0,Qwen/Qwen3-4B,logistic_core_stage10,3,0.910633,0.870515,0.113458,0.367359,2.210713e-02,-0.135585
1,Qwen/Qwen3-4B,raw_verification_stage10,3,0.898671,0.893999,0.168877,1.601189,1.681513e-01,-0.080166
2,Qwen/Qwen3-4B,train_prevalence_stage10,3,0.500000,0.469072,0.249043,0.691233,0.000000e+00,0.000000
3,Qwen/Qwen3-4B,xgb_core_fixed,3,0.924010,0.908439,0.108511,0.350852,2.838616e-02,-0.140533
4,Qwen/Qwen3-4B,xgb_core_minus_consistency,3,0.923559,0.907794,0.108882,0.351823,2.627773e-02,-0.140161
5,Qwen/Qwen3-4B,xgb_core_nested,3,0.923568,0.907865,0.108931,0.352088,2.790713e-02,-0.140112
6,Qwen/Qwen3-4B,xgb_extended,3,0.924466,0.907530,0.108244,0.349224,2.124287e-02,-0.140799
7,google/gemma-3-4b-it,logistic_core_stage10,3,0.822222,0.875019,0.150501,0.473200,1.988744e-02,-0.074295
8,google/gemma-3-4b-it,raw_verification_stage10,3,0.782700,0.867950,0.274458,2.361044,2.741019e-01,0.049661
9,google/gemma-3-4b-it,train_prevalence_stage10,3,0.498166,0.657943,0.224797,0.641858,0.000000e+00,0.000000


## 7. Eşleştirilmiş soru-düzeyi bootstrap: XGBoost ↔ Logistic CORE

`Δ = loss(alternative) − loss(reference)`; **pozitif Δ = referansın daha düşük hatası**, negatif Δ = alternatifin daha düşük hatası. Aynı soru için üç tekrar bir arada örneklenir. Modeller bootstrap örneklemlerinde yeniden eğitilmediğinden aralıklar mevcut OOF tahminlerine koşulludur; final test güven aralığı değildir.

In [8]:
COMPARE = [
    ('logistic_core_stage10', 'xgb_core_nested'),
    ('logistic_core_stage10', 'xgb_core_fixed'),
    ('xgb_core_nested', 'xgb_core_fixed'),
    ('xgb_core_nested', 'xgb_core_minus_consistency'),
    ('xgb_core_nested', 'xgb_extended'),
]
paired_rows = []
for model_idx, (model_id, part) in enumerate(clean.groupby('model_id',sort=True)):
    questions = sorted(part['question_id'].unique())
    assert len(questions) == 970
    index = pd.MultiIndex.from_product([questions, range(len(SEEDS))],
                                       names=['question_id','repeat'])
    def method_matrix(method):
        df = all_oof.loc[(all_oof['model_id']==model_id)&(all_oof['method']==method),
            ['question_id','repeat','manual_is_correct','p_correct']].copy()
        assert not df.duplicated(['question_id','repeat']).any()
        df = df.set_index(['question_id','repeat'])
        if set(df.index) != set(index):
            raise RuntimeError(f'Incomplete OOF: {model_id}/{method}')
        df = df.loc[index]
        p = df['p_correct'].to_numpy(dtype=float).reshape(len(questions),len(SEEDS))
        y = df['manual_is_correct'].to_numpy(dtype=int).reshape(len(questions),len(SEEDS))
        assert np.all(y == y[:,[0]])
        return p,y

    matrices = {method: method_matrix(method) for pair in COMPARE for method in pair}
    first_y = next(iter(matrices.values()))[1]
    assert all(np.array_equal(y,first_y) for p,y in matrices.values())
    rng = np.random.default_rng(BOOTSTRAP_SEED + model_idx)
    idx = rng.integers(0,len(questions),size=(BOOTSTRAP_REPS,len(questions)))

    for reference, alternative in COMPARE:
        rp,y = matrices[reference]
        ap,_ = matrices[alternative]
        brier_delta = ((ap-y)**2 - (rp-y)**2)
        apc = np.clip(ap, EPS, 1-EPS)
        rpc = np.clip(rp, EPS, 1-EPS)
        nll_alt = -(y*np.log(apc)+(1-y)*np.log1p(-apc))
        nll_ref = -(y*np.log(rpc)+(1-y)*np.log1p(-rpc))
        nll_delta = nll_alt - nll_ref
        per_q_brier = brier_delta.mean(axis=1)
        per_q_nll = nll_delta.mean(axis=1)
        b_lo,b_hi = np.percentile(per_q_brier[idx].mean(axis=1),[2.5,97.5])
        n_lo,n_hi = np.percentile(per_q_nll[idx].mean(axis=1),[2.5,97.5])
        row = {
            'model_id':model_id,'reference_method':reference,'alternative_method':alternative,
            'n_unique_questions':len(questions),'n_repeats':len(SEEDS),
            'delta_brier_alt_minus_ref':float(per_q_brier.mean()),
            'delta_brier_ci95_low':float(b_lo),'delta_brier_ci95_high':float(b_hi),
            'delta_nll_alt_minus_ref':float(per_q_nll.mean()),
            'delta_nll_ci95_low':float(n_lo),'delta_nll_ci95_high':float(n_hi),
            'bootstrap_reps':BOOTSTRAP_REPS,
        }
        for rep in range(len(SEEDS)):
            row[f'repeat{rep}_delta_brier'] = float(brier_delta[:,rep].mean())
            row[f'repeat{rep}_delta_nll'] = float(nll_delta[:,rep].mean())
        paired_rows.append(row)

paired11 = pd.DataFrame(paired_rows)
print('PAIRED QUESTION-LEVEL BOOTSTRAP: PASS')
display(paired11[[
    'model_id','reference_method','alternative_method',
    'delta_brier_alt_minus_ref','delta_brier_ci95_low','delta_brier_ci95_high',
    'delta_nll_alt_minus_ref','delta_nll_ci95_low','delta_nll_ci95_high',
    'repeat0_delta_brier','repeat1_delta_brier','repeat2_delta_brier',
]])

PAIRED QUESTION-LEVEL BOOTSTRAP: PASS


,model_id,reference_method,alternative_method,delta_brier_alt_minus_ref,delta_brier_ci95_low,delta_brier_ci95_high,delta_nll_alt_minus_ref,delta_nll_ci95_low,delta_nll_ci95_high,repeat0_delta_brier,repeat1_delta_brier,repeat2_delta_brier
0,Qwen/Qwen3-4B,logistic_core_stage10,xgb_core_nested,-0.004527,-0.010848,0.001497,-0.015271,-0.033741,0.002408,-0.004452,-0.005610,-0.003519
1,Qwen/Qwen3-4B,logistic_core_stage10,xgb_core_fixed,-0.004947,-0.011336,0.001095,-0.016507,-0.035076,0.001156,-0.004452,-0.005610,-0.004779
2,Qwen/Qwen3-4B,xgb_core_nested,xgb_core_fixed,-0.000420,-0.000822,-0.000043,-0.001236,-0.002466,0.000049,0.000000,0.000000,-0.001260
3,Qwen/Qwen3-4B,xgb_core_nested,xgb_core_minus_consistency,-0.000049,-0.000466,0.000354,-0.000265,-0.001333,0.000749,0.000443,-0.000512,-0.000077
4,Qwen/Qwen3-4B,xgb_core_nested,xgb_extended,-0.000687,-0.003382,0.002011,-0.002864,-0.010297,0.004352,-0.000200,-0.000288,-0.001572
5,google/gemma-3-4b-it,logistic_core_stage10,xgb_core_nested,-0.008262,-0.015452,-0.000924,-0.031472,-0.052779,-0.010640,-0.007367,-0.007803,-0.009617
6,google/gemma-3-4b-it,logistic_core_stage10,xgb_core_fixed,-0.009531,-0.016783,-0.002106,-0.034672,-0.055804,-0.013583,-0.007367,-0.011609,-0.009617
7,google/gemma-3-4b-it,xgb_core_nested,xgb_core_fixed,-0.001269,-0.001793,-0.000717,-0.003199,-0.004773,-0.001516,0.000000,-0.003806,0.000000
8,google/gemma-3-4b-it,xgb_core_nested,xgb_core_minus_consistency,-0.000218,-0.000620,0.000188,-0.000451,-0.001461,0.000543,-0.000391,-0.000249,-0.000013
9,google/gemma-3-4b-it,xgb_core_nested,xgb_extended,-0.003234,-0.006518,0.000071,-0.007873,-0.016147,0.000582,-0.003977,-0.003402,-0.002323


## 8. Risk–coverage (geliştirme ön incelemesi)

Coverage: en yüksek confidence'a sahip ilk `%k` cevap kullanıcıya gösterilirse kapsanan soru oranı. Risk: gösterilen cevaplar içindeki yanlış cevap oranı. Bu, bir sistemi düşük güvenli cevapları yanıtsız bırakmaya izin verdiğimizde ne olacağını gösterir. Eşik seçimi **bu OOF eğrisinden sonra final test sonuçlarına göre yapılmamalı**.

In [9]:
risk_rows = []
coverage_points = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]
for (model_id, rep, method), block in all_oof.loc[
    all_oof['method'].isin(['logistic_core_stage10','xgb_core_nested'])
].groupby(['model_id','repeat','method'],sort=True):
    block = block.sort_values('question_id')
    y = block['manual_is_correct'].astype(int).to_numpy()
    p = block['p_correct'].to_numpy(dtype=float)
    selected_order = np.argsort(-p, kind='mergesort')
    for requested in coverage_points:
        take = max(1, int(np.ceil(requested * len(y))))
        risk_rows.append({
            'model_id':model_id,'repeat':int(rep),'method':method,
            'requested_coverage':requested,'n_covered':take,
            'realized_coverage':float(take/len(y)),
            'empirical_risk':float(1-y[selected_order[:take]].mean()),
            'empirical_accuracy':float(y[selected_order[:take]].mean()),
        })
risk11 = pd.DataFrame(risk_rows)
assert len(risk11)==3*len(SEEDS)*2*len(coverage_points)
print('RISK-COVERAGE DEVELOPMENT AUDIT: PASS')
display(risk11.loc[(risk11['repeat']==0)&(risk11['requested_coverage'].isin([0.2,0.5,1.0])),
                   ['model_id','method','requested_coverage','n_covered','empirical_risk']])

RISK-COVERAGE DEVELOPMENT AUDIT: PASS


,model_id,method,requested_coverage,n_covered,empirical_risk
1,Qwen/Qwen3-4B,logistic_core_stage10,0.2,194,0.118557
4,Qwen/Qwen3-4B,logistic_core_stage10,0.5,485,0.183505
9,Qwen/Qwen3-4B,logistic_core_stage10,1.0,970,0.530928
11,Qwen/Qwen3-4B,xgb_core_nested,0.2,194,0.025773
14,Qwen/Qwen3-4B,xgb_core_nested,0.5,485,0.185567
19,Qwen/Qwen3-4B,xgb_core_nested,1.0,970,0.530928
61,google/gemma-3-4b-it,logistic_core_stage10,0.2,194,0.072165
64,google/gemma-3-4b-it,logistic_core_stage10,0.5,485,0.142268
69,google/gemma-3-4b-it,logistic_core_stage10,1.0,970,0.341237
71,google/gemma-3-4b-it,xgb_core_nested,0.2,194,0.041237


## 9. Immutable Stage 11 artifacts + manifest

Çalıştırmadan önce yukarıdaki metrik/CI çıktılarını kontrol et. Bu hücre ana Stage 10 dosyalarını değiştirmez. Stage 11 çıktıları kendi hash'li klasörüne yazılır. Mevcut bir dosya farklı içerikle çakışırsa durur.

In [10]:
def immutable_parquet(df,path):
    if path.exists():
        old = pd.read_parquet(path)
        pd.testing.assert_frame_equal(
            old.reset_index(drop=True), df.reset_index(drop=True), check_exact=True
        )
        return
    temp = path.with_name(path.name+'.tmp.parquet')
    df.to_parquet(temp,index=False,engine='pyarrow')
    os.replace(temp,path)

immutable_csv(inner_scores, PATHS['inner_cv'])
immutable_csv(selection_df, PATHS['selection'])
immutable_parquet(xgb_oof, PATHS['oof'])
immutable_csv(metrics11, PATHS['metrics'])
immutable_csv(summary11, PATHS['summary'])
immutable_csv(paired11, PATHS['paired'])
immutable_csv(risk11, PATHS['risk_coverage'])

manifest = {
    'stage':'11_nonlinear_confidence_dev',
    'dataset_version':VERSION,
    'assembly_version':ASSEMBLY,
    'stage08_manifest_sha256':sha_file(M08),
    'stage05_repair_manifest_sha256':sha_file(REPAIR),
    'stage10_manifest_sha256':sha_file(M10),
    'stage10_frozen_oof_sha256':sha_file(stage10_oof_path),
    'n_models':3,'n_unique_questions_per_model':970,
    'n_outer_folds_per_model':N_SPLITS*len(SEEDS),
    'n_inner_candidate_evaluations':len(inner_scores),
    'n_new_oof_rows':len(xgb_oof),
    'primary_comparison':'xgb_core_nested vs frozen Stage10 logistic_core',
    'output_files':{
        name:{'path':str(path),'sha256':sha_file(path)}
        for name,path in PATHS.items() if name != 'manifest'
    },
    'status':'development_oof_not_independent_final_test',
}
immutable_bytes(PATHS['manifest'],json.dumps(
    manifest,ensure_ascii=False,sort_keys=True,indent=2
).encode('utf-8'))
print('STAGE 11 DEVELOPMENT DONDURULDU')
print('Manifest:', PATHS['manifest'])
print('Output:',OUT)

STAGE 11 DEVELOPMENT DONDURULDU
Manifest: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage11/validated_v007/stage10_ada008b24140_xgb_nested_v1/stage11_dev_manifest_v1.json
Output: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage11/validated_v007/stage10_ada008b24140_xgb_nested_v1


## Stage 11 tamamlanma kriterleri

- [ ] `Stage08 repair-aware provenance: PASS` ve `Stage10 frozen SHA/provenance: PASS`
- [ ] `STAGE 10 / STAGE 11 SAME-FOLD + LABEL CONTRACT: PASS`
- [ ] `XGBOOST HELPER SMOKE TEST: PASS`
- [ ] `XGBOOST OOF COVERAGE + STAGE10 SAME-FOLD: PASS`
- [ ] `STAGE10 BASELINE REPRODUCTION: PASS`
- [ ] `PAIRED QUESTION-LEVEL BOOTSTRAP: PASS`
- [ ] `RISK-COVERAGE DEVELOPMENT AUDIT: PASS`
- [ ] `STAGE 11 DEVELOPMENT DONDURULDU`

**Yorum sırası:** Önce XGBoost CORE nested ile frozen Logistic CORE'un *paired* Brier/NLL farkını, sonra AUROC/AUPRC ve ECE'yi, sonra consistency / EXTENDED ikincil ablation'larını inceleyeceğiz. Ana test seti henüz oluşturulmadı; model seçiminde kullandığımız 1000 sorudan yeni bir random split yapıp onu bağımsız final test olarak adlandırmayacağız.